# DBSCAN: Finding Clusters by Density

**COMP 034 Introduction to Data Science · BSIT 3-3 · Group Report**

This notebook follows our report slide by slide. It uses the same story: finding **road-accident hotspots** on a map.

| Part | What you will see | Slides |
| --- | --- | --- |
| 0 | Setup | – |
| 1 | The problem: a map full of accident dots | 3 |
| 2 | DBSCAN vs. K-Means | 5 |
| 3 | The two settings: ε and MinPts | 6–7 |
| 4 | Worked example: find the core points | 8, 10 |
| 5 | Build DBSCAN ourselves in 4 steps | 9 |
| 6 | The same thing with scikit-learn | 11 |
| 7 | What the result tells us | 12 |
| 8 | Change the settings, change the result | 13 |
| 9 | Choosing ε with a k-distance graph | 13 |
| 10 | Knowledge check: core, border, or noise? | 16 |
| 11 | A limitation: hotspots with different densities | 15 |
| 12 | Try it yourself | – |

**How to run:** pick the `.venv` kernel (see `README.md`), then click **Run All**.

All map points in this notebook are made up for practice. Only the EDSA and Commonwealth numbers in Part 1 are real.

## Part 0 · Setup

We load the tools we need and pick the same colors as our slides.

In [ ]:
# numpy: works with lists of numbers, like our map coordinates
import numpy as np

# pandas: shows results as neat tables
import pandas as pd

# matplotlib: draws our maps and charts
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

# scikit-learn: has DBSCAN and K-Means ready to use
from sklearn.cluster import DBSCAN, KMeans
from sklearn.datasets import make_moons
from sklearn.neighbors import NearestNeighbors

# Colors from our slides
TEAL = "#136F7C"     # cluster 1
NAVY = "#0F2D3F"     # cluster 2
AQUA = "#8ACFD5"     # light teal for border points
ORANGE = "#D2692B"   # noise
GRAY = "#7A8C97"     # points we have not grouped yet
MIST = "#EAF3F4"     # map background

# A few more colors in case DBSCAN finds more than 2 clusters
CLUSTER_COLORS = [TEAL, NAVY, "#6C4BA6", "#2E7D32", "#B8860B", "#8E3B5E"]

# Make the charts a little sharper
plt.rcParams["figure.dpi"] = 110

print("Setup done. Ready to go!")

### Two small helpers

We will draw many maps, so we write the drawing code once.

- `point_types` tells us if each point is a **core**, **border**, or **noise** point.
- `draw_map` draws the points: dark dot = core, light dot = border, orange ring = noise.

In [ ]:
def point_types(model, n_points):
    # Start by calling every point "border"
    types = np.array(["border"] * n_points, dtype=object)

    # DBSCAN remembers which points were core points
    types[model.core_sample_indices_] = "core"

    # DBSCAN gives noise points the label -1
    types[model.labels_ == -1] = "noise"
    return types


def draw_map(ax, X, labels, types, title, names=None):
    ax.set_facecolor(MIST)

    for i, (x, y) in enumerate(X):
        if types[i] == "noise":
            # Noise: an orange ring
            ax.scatter(x, y, s=90, facecolors="none", edgecolors=ORANGE, linewidths=2)
        else:
            color = CLUSTER_COLORS[labels[i] % len(CLUSTER_COLORS)]
            if types[i] == "core":
                # Core: a solid dot in the cluster's color
                ax.scatter(x, y, s=90, color=color)
            else:
                # Border: a light dot with the cluster's color as the outline
                ax.scatter(x, y, s=90, color=AQUA, edgecolors=color, linewidths=2)

        # Write the point's name (A, B, C...) next to it
        if names is not None:
            ax.annotate(names[i], (x, y), xytext=(7, 7), textcoords="offset points",
                        fontsize=10, fontweight="bold", color=NAVY)

    ax.set_title(title)
    ax.set_aspect("equal")
    ax.set_axisbelow(True)   # keep the grid lines behind the dots
    ax.grid(color="white", linewidth=1)

## Part 1 · The problem: where are the danger zones? (Slide 3)

In 2024, the MMDA recorded **8,873** road accidents on EDSA and **5,091** on Commonwealth Avenue (GMA News, June 26, 2025).

Accidents do not happen evenly. Some spots get hit again and again. These are **hotspots**.

Let's make a pretend city map. Each dot is one accident. The numbers are in **meters**.

In [ ]:
# A "seed" makes the random numbers the same every time we run this
rng = np.random.default_rng(2026)

# Three busy spots in our pretend city: (center x, center y), how many accidents, how spread out
hotspots = [
    ((600, 1100), 40, 60),
    ((1500, 700), 35, 70),
    ((1050, 350), 25, 50),
]

groups = []
for center, how_many, spread in hotspots:
    # Scatter accidents around each busy spot
    groups.append(rng.normal(center, spread, size=(how_many, 2)))

# 40 more accidents anywhere on a 2 km x 1.5 km map
groups.append(rng.uniform([0, 0], [2000, 1500], size=(40, 2)))

# Put all the accidents together into one table of x and y
city = np.vstack(groups)
print("Number of accidents on the map:", len(city))

fig, ax = plt.subplots(figsize=(8, 6))
ax.set_facecolor(MIST)
ax.scatter(city[:, 0], city[:, 1], s=25, color=NAVY)
ax.set_title("Pretend city map: each dot = one accident")
ax.set_xlabel("meters")
ax.set_ylabel("meters")
ax.set_aspect("equal")
plt.show()

Our eyes can see a few crowded areas. But the data has **no labels**. Nobody told the computer which dot belongs to which hotspot.

Finding groups without labels is **unsupervised learning**. The job itself is called **clustering**.

## Part 2 · DBSCAN vs. K-Means (Slide 5)

Same 84 points, two methods. The data has two curved shapes (like two bananas) and 4 lonely points.

In [ ]:
# Make 80 points in two curved shapes
moons, _ = make_moons(n_samples=80, noise=0.06, random_state=7)

# Add 4 lonely points far from the shapes
lonely = np.array([[-0.9, 1.15], [2.0, -0.6], [-0.7, -0.55], [2.1, 1.1]])
moons = np.vstack([moons, lonely])

# K-Means: we MUST tell it how many groups to make (k = 2)
kmeans_labels = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(moons)

# DBSCAN: we only give the two settings, not the number of groups
moons_model = DBSCAN(eps=0.25, min_samples=4).fit(moons)
moons_types = point_types(moons_model, len(moons))

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: K-Means colors every point, even the lonely ones
left.set_facecolor(MIST)
left.scatter(moons[:, 0], moons[:, 1], s=40,
             color=[TEAL if g == 0 else NAVY for g in kmeans_labels])
left.set_title("K-Means (k = 2): cuts the curves, no noise")

# Right: DBSCAN follows the curves and rings the lonely points in orange
draw_map(right, moons, moons_model.labels_, moons_types, "DBSCAN: follows the curves, finds noise")
right.set_aspect("auto")
plt.show()

print("Noise points DBSCAN found:", (moons_model.labels_ == -1).sum())

**What to notice**

- K-Means needs **k** first, likes round groups, and forces **every** point into a group.
- DBSCAN finds the number of groups **by itself**, follows **any shape**, and marks lonely points as **noise**.

## Part 3 · The two settings: ε and MinPts (Slides 6–7)

Think of lunch break at the PUP Lagoon.

- **ε (epsilon)** is your **arm's reach**: the radius of the circle around a point. Anyone inside is your neighbor.
- **MinPts** is **how many people make a crowd**: the fewest points inside the circle, **counting the point itself**.

In our accident example: **ε = 150 meters** and **MinPts = 4**.

To know who is "inside the circle", we measure distance as a straight line (**Euclidean distance**):

$$d = \sqrt{(x_2 - x_1)^2 + (y_2 - y_1)^2}$$

In [ ]:
def distance(p, q):
    # Straight-line distance between two points, like using a ruler on the map
    return np.sqrt((q[0] - p[0]) ** 2 + (q[1] - p[1]) ** 2)


# Two accidents 300 m apart going right, 400 m apart going up
print("Distance:", distance((0, 0), (300, 400)), "meters")

# Is that inside a 150 m circle?
print("Inside the 150 m circle?", distance((0, 0), (300, 400)) <= 150)

## Part 4 · Worked example: find the core points (Slides 8 and 10)

These are the same 14 accidents, **A to N**, from our slides. Each grid box is 100 meters.

A point is a **core point** if it has **at least MinPts points within ε**, counting itself.

In [ ]:
# The 14 accidents from our slides (x, y) in meters
accidents = {
    "A": (200, 300), "B": (300, 300), "C": (200, 400), "D": (300, 400),
    "E": (240, 200), "F": (430, 360), "G": (700, 200), "H": (800, 200),
    "I": (750, 290), "J": (760, 110), "K": (930, 250), "L": (560, 500),
    "M": (80, 80),   "N": (960, 500),
}

names = list(accidents.keys())
X = np.array(list(accidents.values()), dtype=float)

# Our two settings
EPS = 150      # epsilon: the circle's radius, in meters
MIN_PTS = 4    # MinPts: the fewest points in the circle to be "crowded" (counts itself)

# Make a 14 x 14 table: the distance from every point to every other point
distances = np.array([[distance(p, q) for q in X] for p in X])

# For each point, count how many points are inside its circle (itself included)
counts = (distances <= EPS).sum(axis=1)

# Show it as a table, like the one on Slide 10
table = pd.DataFrame({
    "Point": names,
    "Points within 150 m": counts,
    "Core?": np.where(counts >= MIN_PTS, "Yes", "No"),
})
table

In [ ]:
# Which points are inside B's circle?
b = names.index("B")
inside_b = [names[j] for j in range(len(names)) if distances[b, j] <= EPS]
print("Inside B's circle:", inside_b, "->", len(inside_b), "points, so B is a core point")

core_points = [names[i] for i in range(len(names)) if counts[i] >= MIN_PTS]
print("Core points:", core_points)

In [ ]:
# Draw the map with B's 150 m circle, like Slide 10
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.set_facecolor(MIST)
ax.scatter(X[:, 0], X[:, 1], s=90, color=NAVY)

for name, (x, y) in accidents.items():
    ax.annotate(name, (x, y), xytext=(7, 7), textcoords="offset points",
                fontsize=11, fontweight="bold", color=NAVY)

# The dashed circle shows B's neighborhood
ax.add_patch(Circle(accidents["B"], EPS, fill=True, facecolor=TEAL, alpha=0.12,
                    edgecolor=TEAL, linestyle="--", linewidth=2))

ax.set_xlim(0, 1050)
ax.set_ylim(0, 600)
ax.set_aspect("equal")
ax.set_xticks(range(0, 1100, 100))
ax.set_yticks(range(0, 700, 100))
ax.set_axisbelow(True)   # keep the grid lines behind the dots
ax.grid(color="white", linewidth=1)
ax.set_title("B's 150 m circle holds 6 points: A, B, C, D, E, F")
plt.show()

## Part 5 · Build DBSCAN ourselves in 4 steps (Slide 9)

Before using the ready-made tool, let's write the 4 steps from our slide in plain Python:

1. **Pick** a point we have not visited yet.
2. **Count** its neighbors inside the circle (itself included).
3. **Core?** Start a cluster, add its neighbors, and keep growing from every new core point.
4. **Not core?** Mark it as noise **for now**. It can still join a cluster later as a border point.

In [ ]:
def my_dbscan(X, eps, min_pts):
    n = len(X)
    NOISE = -1
    labels = [None] * n          # None means "not visited yet"
    cluster_id = -1              # no clusters yet

    def neighbors_of(i):
        # Every point inside i's circle, including i itself
        return [j for j in range(n) if distance(X[i], X[j]) <= eps]

    for i in range(n):
        # Step 1: pick a point we have not visited yet
        if labels[i] is not None:
            continue

        # Step 2: count its neighbors
        neighbors = neighbors_of(i)

        # Step 4: not enough neighbors? Mark it as noise, for now
        if len(neighbors) < min_pts:
            labels[i] = NOISE
            continue

        # Step 3: it is a core point, so start a new cluster
        cluster_id += 1
        labels[i] = cluster_id
        to_check = list(neighbors)

        # Keep growing the cluster like a chain
        while to_check:
            j = to_check.pop(0)

            # A "noise" point that a core point can reach becomes a border point
            if labels[j] == NOISE:
                labels[j] = cluster_id
            if labels[j] is not None:
                continue

            # Add this point to the cluster
            labels[j] = cluster_id

            # If this point is ALSO a core point, its neighbors join too
            j_neighbors = neighbors_of(j)
            if len(j_neighbors) >= min_pts:
                to_check.extend(j_neighbors)

    return np.array(labels)


my_labels = my_dbscan(X, EPS, MIN_PTS)
pd.DataFrame({"Point": names, "Cluster (-1 = noise)": my_labels}).T

## Part 6 · The same thing with scikit-learn (Slide 11)

In real projects we use the ready-made `DBSCAN` from scikit-learn.

- `eps` is ε.
- `min_samples` is MinPts. In scikit-learn it **includes the point itself**, the same as our slides.
- Noise points get the label **-1**.

In [ ]:
# Create DBSCAN with our two settings, then let it look at the 14 accidents
model = DBSCAN(eps=EPS, min_samples=MIN_PTS).fit(X)
types = point_types(model, len(X))

result = pd.DataFrame({
    "Point": names,
    "Cluster": model.labels_,   # 0 = Cluster 1, 1 = Cluster 2, -1 = noise
    "Type": types,
})
print("Same answer as our own 4-step version?", (model.labels_ == my_labels).all())
result

In [ ]:
# Draw the final clusters, like Slide 11
fig, ax = plt.subplots(figsize=(9, 5.5))
draw_map(ax, X, model.labels_, types, "Cluster 1 (teal), Cluster 2 (navy), noise (orange rings)", names)
ax.set_xlim(0, 1050)
ax.set_ylim(0, 600)
plt.show()

## Part 7 · What the result tells us (Slide 12)

Let's turn the labels into a short report, like we would give the city.

In [ ]:
# Go through each hotspot (cluster numbers 0, 1, ...). -1 is noise, so we skip it here
for cluster in sorted(set(model.labels_) - {-1}):
    # Pick the names of the points in this hotspot
    members = [names[i] for i in range(len(names)) if model.labels_[i] == cluster]
    core = [names[i] for i in range(len(names)) if model.labels_[i] == cluster and types[i] == "core"]
    print(f"Hotspot {cluster + 1}: {len(members)} accidents {members}, core points {core}")

# The points labeled -1 are the lone accidents
lone = [names[i] for i in range(len(names)) if model.labels_[i] == -1]
print(f"Noise: {len(lone)} lone accidents {lone}. Keep recording them.")

**14 dots became 2 hotspots and 3 lone accidents.** Hotspot 1 should get help first (traffic enforcers, lights, road signs), then Hotspot 2.

## Part 8 · Change the settings, change the result (Slide 13)

Same 14 points. We only change ε.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

for ax, eps in zip(axes, [90, 150, 300]):
    m = DBSCAN(eps=eps, min_samples=MIN_PTS).fit(X)
    t = point_types(m, len(X))

    # Count clusters (ignore -1, which is noise) and noise points
    n_clusters = len(set(m.labels_) - {-1})
    n_noise = (m.labels_ == -1).sum()

    draw_map(ax, X, m.labels_, t, f"ε = {eps} m: {n_clusters} cluster(s), {n_noise} noise")
    ax.set_xlim(0, 1050)
    ax.set_ylim(0, 600)

plt.show()

- **Too small (90 m):** nobody has enough neighbors, so all 14 are noise.
- **Just right (150 m):** our 2 hotspots and 3 noise points.
- **Too big (300 m):** everything melts into 1 giant cluster.

Now let's keep ε = 150 m and change **MinPts** instead.

In [ ]:
rows = []
for min_pts in [3, 4, 5, 6]:
    m = DBSCAN(eps=EPS, min_samples=min_pts).fit(X)
    rows.append({
        "MinPts": min_pts,
        "Clusters": len(set(m.labels_) - {-1}),
        "Noise points": (m.labels_ == -1).sum(),
    })

# Higher MinPts = stricter: you need a bigger crowd
pd.DataFrame(rows)

At **MinPts = 6**, no point in the second hotspot has 6 points around it, so Cluster 2 disappears and its points become noise. Higher MinPts is **stricter**.

## Part 9 · Choosing ε with a k-distance graph (Slide 13)

How do we pick ε for a bigger map? A common trick:

1. For every point, measure the distance to its k-th nearest neighbor (k = MinPts, counting the point itself).
2. Sort these distances from small to big and draw them.
3. Look for the **bend** where the line suddenly shoots up. That distance is a good ε.

Points on the low, flat part are in crowded areas. Points after the bend are the lonely ones.

In [ ]:
# Ask for each point's 4 nearest points (the first one is the point itself)
finder = NearestNeighbors(n_neighbors=MIN_PTS).fit(city)
dist_to_neighbors, _ = finder.kneighbors(city)

# Keep the distance to the 4th one, then sort from small to big
k_distances = np.sort(dist_to_neighbors[:, -1])

CITY_EPS = 100   # our pick, read from the bend in the graph

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(k_distances, color=TEAL, linewidth=2.5)
ax.axhline(CITY_EPS, color=ORANGE, linestyle="--", label=f"our ε = {CITY_EPS} m")
ax.set_xlabel("points, sorted by distance")
ax.set_ylabel("distance to 4th nearest point (m)")
ax.set_title("k-distance graph: pick ε near the bend")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

In [ ]:
# Now run DBSCAN on the whole pretend city with ε = 100 m and MinPts = 4
city_model = DBSCAN(eps=CITY_EPS, min_samples=MIN_PTS).fit(city)
city_types = point_types(city_model, len(city))

fig, ax = plt.subplots(figsize=(8, 6))
draw_map(ax, city, city_model.labels_, city_types, "Hotspots found in the pretend city")
ax.set_xlabel("meters")
ax.set_ylabel("meters")
plt.show()

# How many accidents are in each hotspot?
for cluster in sorted(set(city_model.labels_)):
    size = (city_model.labels_ == cluster).sum()
    name = "Noise (lone accidents)" if cluster == -1 else f"Hotspot {cluster + 1}"
    print(f"{name}: {size}")

## Part 10 · Knowledge check: core, border, or noise? (Slide 16)

Use ε = 150 m and MinPts = 4. Each mini map has one point we care about (P, Q, or R) at (0, 0).

Guess first, then run the cell.

In [ ]:
mini_maps = {
    # P has 4 neighbors close by (5 points counting itself)
    "P": [(0, 0), (-90, -70), (110, -60), (-100, 80), (95, 75)],
    # Q is alone; the other points are far away
    "Q": [(0, 0), (280, 90), (330, -110), (380, 40)],
    # R has 2 neighbors. One of them (at 120, 15) has its own crowd, so it is a core point
    "R": [(0, 0), (-70, 70), (120, 15), (230, -60), (215, 110), (260, 40)],
}

for name, points in mini_maps.items():
    m = DBSCAN(eps=EPS, min_samples=MIN_PTS).fit(np.array(points, dtype=float))
    t = point_types(m, len(points))

    # The point we care about is always the first one, at (0, 0)
    count = (np.array([distance(points[0], q) for q in points]) <= EPS).sum()
    print(f"{name}: {count} point(s) within 150 m -> {t[0].upper()} point")

- **P** has 5 points, and 5 ≥ 4, so it is a **core** point.
- **Q** has only itself and no core point nearby, so it is **noise**.
- **R** has only 3 points, so it is not core. But one neighbor is a core point, so R is a **border** point.

**Core** = in the crowd. **Border** = at the edge of a crowd. **Noise** = alone.

## Part 11 · A limitation: hotspots with different densities (Slide 15)

One packed hotspot and one spread-out hotspot, side by side. Can one ε fit both?

In [ ]:
rng2 = np.random.default_rng(7)
packed = rng2.normal((300, 350), 25, size=(60, 2))      # very crowded hotspot
spread = rng2.normal((700, 350), 110, size=(30, 2))     # spread-out hotspot
extra = rng2.uniform((0, 0), (1100, 700), size=(20, 2))  # random lone accidents
mixed = np.vstack([packed, spread, extra])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, eps in zip(axes, [40, 160]):
    m = DBSCAN(eps=eps, min_samples=MIN_PTS).fit(mixed)
    t = point_types(m, len(mixed))
    n_clusters = len(set(m.labels_) - {-1})
    draw_map(ax, mixed, m.labels_, t, f"ε = {eps} m: {n_clusters} cluster(s), {(m.labels_ == -1).sum()} noise")
plt.show()

- **Small ε (40 m):** the packed hotspot is found, but the spread-out one breaks into pieces and noise.
- **Big ε (160 m):** both hotspots melt into one cluster, and some lone accidents get swallowed too.

No single ε fits both. Other limits from our slides:

- Very **sensitive** to ε and MinPts (Part 8).
- **Many columns** (high dimensions) make distances less useful.
- Features must be on the **same scale**. Meters and pesos in one table would confuse the distances, so scale them first.

## Part 12 · Try it yourself

Change the two numbers below and run the cell. Watch how the hotspots change.

In [ ]:
MY_EPS = 100      # try 60, 100, 150, 250
MY_MIN_PTS = 4    # try 3, 4, 8, 12

m = DBSCAN(eps=MY_EPS, min_samples=MY_MIN_PTS).fit(city)
t = point_types(m, len(city))
n_clusters = len(set(m.labels_) - {-1})

fig, ax = plt.subplots(figsize=(8, 6))
draw_map(ax, city, m.labels_, t,
         f"ε = {MY_EPS} m, MinPts = {MY_MIN_PTS}: {n_clusters} hotspot(s), {(m.labels_ == -1).sum()} noise")
plt.show()

## Key terms for the exam

- **DBSCAN**: Density-Based Spatial Clustering of Applications with Noise (Ester, Kriegel, Sander & Xu, 1996).
- **ε-neighborhood**: all points within distance ε of a point.
- **MinPts**: the fewest points in the ε-neighborhood, counting the point itself, for a core point.
- **Core point**: at least MinPts points within ε.
- **Border point**: fewer than MinPts, but within ε of a core point.
- **Noise point**: not a core point and not within ε of any core point.
- **Directly density-reachable**: within ε of a core point.
- **Density-reachable**: linked to a core point through a chain of core points.

## Sources

- Ester, M., Kriegel, H.-P., Sander, J., & Xu, X. (1996). A density-based algorithm for discovering clusters in large spatial databases with noise. *Proceedings of KDD-96*, 226–231.
- Schubert, E., Sander, J., Ester, M., Kriegel, H.-P., & Xu, X. (2017). DBSCAN revisited, revisited. *ACM Transactions on Database Systems, 42*(3), Article 19.
- scikit-learn developers. `sklearn.cluster.DBSCAN` documentation.
- Serquiña, M. C. (2025, June 26). EDSA logs most road accidents in NCR in 2024. *GMA News*.